# face-cloak: run one experiment on Google Colab

This notebook is a thin wrapper around the `facecloak` package in this repo. All logic lives in `src/facecloak/`,
so the result is identical to running `python scripts/run_experiment.py --config ...` on a workstation.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Outputs go to `runs/` inside the cloned repo (`/content/face-cloak/runs/`), exactly as on a workstation:

```
runs/_shared/images/<id>/clean.png       preprocessed inputs (shared by all experiments)
runs/_shared/masks/                       MediaPipe face masks
runs/_shared/inpaint_<hash>/              clean / random-noise inpaintings (reused by every experiment)
runs/<EXP>/cloak_out/                     cloaks, manifest.csv, figures
runs/<EXP>/inpaint_out/                   inpaintings, results.csv, summary.csv, paired.csv, figures
```

The Colab disk is wiped when the runtime is recycled. The last cell zips `runs/<EXP>` for download.
Every stage resumes from what is already on disk, so a disconnect inside one session only costs the batch in progress.
The identity threshold τ is fixed in `configs/base.yaml` (`eval.tau`), so every clone uses the same value.

In [ ]:
!nvidia-smi -L

## 1. Get the code, install

In [ ]:
REPO_URL = "https://github.com/dinhieufam/face-cloak.git"
REPO_DIR = "/content/face-cloak"
BRANCH   = "main"

import os
if not os.path.exists(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# A CPU-only onnxruntime would shadow the GPU build that uniface[gpu] installs, so remove it first.
!pip uninstall -y -q onnxruntime
!pip install -q -e "{REPO_DIR}[eval]"

import sys
sys.path.insert(0, f"{REPO_DIR}/src")   # editable installs are only picked up after a restart otherwise

## 2. Pick the experiment

| EXP | change vs. baseline |
|---|---|
| `E0_baseline` | none: PhotoGuard PGD, gray target (also runs the random-noise control) |
| `E1_pattern_target` | target image = `assets/targets/pattern.png` |
| `E2_mask_aware` | loss on the attacker's view (face on gray) + mask jitter |
| `E3_momentum` | MI-FGSM, momentum 0.9 |
| `E4_untargeted` | push away from E(x) instead of pulling to a target |

`OVERRIDES` takes dotted keys, e.g. `["quick_test=true"]` (2 images, 50 iterations, ~3 min) or `["cloak.eps=[8,16]"]`.
See `configs/base.yaml` for every key and `docs/experiments.md` for the protocol.

In [ ]:
EXP       = "E0_baseline"
OVERRIDES = []

from facecloak import env
from facecloak.config import get_paths, load_config
from facecloak.viz import show

cfg   = load_config(f"{REPO_DIR}/configs/experiments/{EXP}.yaml", OVERRIDES)
paths = get_paths(cfg)
print(env.describe())
print("experiment:", cfg["exp_name"], "->", paths.exp_dir)

### Kaggle credentials

FFHQ images are fetched one file at a time from Kaggle. If you get a 401/403, add `KAGGLE_USERNAME` and `KAGGLE_KEY`
in the 🔑 Secrets panel and enable notebook access (see `docs/setup.md`).

In [ ]:
from google.colab import userdata
for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        pass  # public downloads often work without credentials

## 3. Identity threshold τ

τ is fixed once for the whole team and stored in `configs/base.yaml` (`eval.tau`, from `scripts/calibrate_tau.py`,
FAR 1% on ~300 FFHQ faces). The cell below only runs the calibration if it is missing.

In [ ]:
if cfg["eval"].get("tau") is not None:
    print("τ =", cfg["eval"]["tau"], "(configs/base.yaml)")
else:
    !cd {REPO_DIR} && python scripts/calibrate_tau.py --config configs/base.yaml

## 4. Stage 1: cloak (PGD on the SD VAE encoder)

In [ ]:
from facecloak.cloak import run_cloak, summarize_cloak

manifest = run_cloak(cfg)
display(summarize_cloak(manifest))
for f in sorted((paths.cloak_dir / "figures").glob("*.png")):
    show(f)

## 5. Stage 2: inpainting attack (SD 1.5)

In [ ]:
from facecloak.inpaint import run_inpaint

gens = run_inpaint(cfg)

## 6. Stage 3: ArcFace evaluation

In [ ]:
from facecloak.evaluate import run_eval

out = run_eval(cfg)
print(f"τ = {out['tau']:.3f}  ({out['tau_info']['tau_source']})")
display(out["summary"])
display(out["paired"])
for f in sorted((paths.inpaint_dir / "figures").glob("*.png")):
    show(f)

## 7. Compare all finished experiments

In [ ]:
from facecloak.evaluate import compare_experiments

table = compare_experiments(paths.root)
table.to_csv(paths.root / "comparison.csv", index=False)
display(table)

## 8. Download the results

Zips `runs/<EXP>` plus `runs/_shared` (inputs, masks, clean/random inpaintings) and `runs/comparison.csv`.

In [ ]:
!cd {REPO_DIR} && zip -q -r /content/{cfg['exp_name']}.zip runs/{cfg['exp_name']} runs/_shared runs/comparison.csv
from google.colab import files
files.download(f"/content/{cfg['exp_name']}.zip")